# Reasoning System

In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

## Datasets

### RavenDataset

In [3]:
import random

import torch
from torch.utils.data import DataLoader

from hri_gnn.datasets import RavenDataset, raven_flat_collate
from hri_gnn.config.const import Configuration

torch.manual_seed(92)
random.seed(92)

DATASET_NAME = 'raven-100'
PERCENTAGE_LIMIT = 0.1
BATCH_SIZE = 10
SPLIT = 'test'
active_configs = [Configuration.IN_DISTRIBUTE_FOUR_OUT_CENTER_SINGLE.value]

test_dataset_raven_100 = RavenDataset(dataset= DATASET_NAME, 
                                       split='test', 
                                       percentage_limit=PERCENTAGE_LIMIT, 
                                       configs=active_configs)

test_dataloader = DataLoader(dataset=test_dataset_raven_100, 
                              batch_size=BATCH_SIZE, 
                              shuffle= (SPLIT == 'train'), 
                              collate_fn=raven_flat_collate)

### PreComputedRavenDataset

In [4]:
import random

import torch
from torch.utils.data import DataLoader

from hri_gnn.datasets.precomputed import PrecomputedRavenDataset, precomputed_collate
from hri_gnn.config.const import Configuration

torch.manual_seed(92)
random.seed(92)

DATASET_NAME = 'raven-100'
PERCENTAGE_LIMIT = 0.1
BATCH_SIZE = 10
SPLIT = 'test'
active_configs = [Configuration.IN_DISTRIBUTE_FOUR_OUT_CENTER_SINGLE.value]

test_precomputed_dataset = PrecomputedRavenDataset(dataset=DATASET_NAME,
                                                   split=SPLIT,
                                                   percentage_limit=PERCENTAGE_LIMIT,
                                                   configs=active_configs)

test_precomputed_dataloader = DataLoader(dataset=test_precomputed_dataset,
                                         batch_size=BATCH_SIZE,
                                         shuffle= SPLIT == 'train',
                                         collate_fn=precomputed_collate,
                                         num_workers=4,
                                         persistent_workers=True,
                                         pin_memory=True)

#### RavenCnnEncoder

needed in the `GraphBuilder` to build graphs on the fly during the training session

In [5]:
from hri_gnn.models import RavenCnnEncoder
from hri_gnn.utils import paths
# load the saved model 
output_path = paths.get_output_path()
raven_encoder = RavenCnnEncoder()
raven_encoder.load_state_dict(torch.load(f=output_path/'models'/'raven_cnn_encoder_final.pth'))

<All keys matched successfully>

#### Original Raw `BATCH`

In [6]:
from itertools import islice
batch = list(islice(test_dataloader,1))[0]
conf_prblm_batch = torch.stack([batch[5],batch[6], batch[8]]).unique(dim=-1)
numb_prblm= conf_prblm_batch.shape[1]
zipped = zip(conf_prblm_batch[0].tolist(), conf_prblm_batch[1].tolist(), conf_prblm_batch[2].tolist())
problem_ids = [(i,j,k) for i,j,k in zipped] 
print(f"batch problems count: {numb_prblm}, (cnfg_id, prblm_id, btc_id): {problem_ids}")

print(f"dataset length: {len(test_dataset_raven_100)}")

batch problems count: 2, (cnfg_id, prblm_id, btc_id): [(4, 8, 1), (4, 79, 0)]
dataset length: 2


#### Precomputed `BATCH`

In [7]:
from hri_gnn.models.gnn_encoder import group_layouts_by_panel
from collections import defaultdict
precomputed_count = 0
problems_graphs = defaultdict(list)
precomputed_batch = list(islice(test_precomputed_dataloader,1))[0]

pyg_batch, all_layout_keys, all_config_ids, all_problem_ids = precomputed_batch
problems = {(all_config_ids[i], all_problem_ids[i]) for i in range(len(all_problem_ids))}

n_problems = pyg_batch.num_graphs // len(range(8, 16))
print(f"Number of problems in the batch: {n_problems}")
print(f"Number of Graphs in the batch : {pyg_batch.num_graphs}, each problem have 8 candidate graphs")
for problem in problems :
    print(f" * problem: {problem}")
    graphs = [str(problem)+f'_C{i}' for i in range(8,16) ]
    print(f" * Graphs:\n - {graphs}")

print(f"dataset length: {len(test_precomputed_dataset)}")

Number of problems in the batch: 2
Number of Graphs in the batch : 16, each problem have 8 candidate graphs
 * problem: (4, 79)
 * Graphs:
 - ['(4, 79)_C8', '(4, 79)_C9', '(4, 79)_C10', '(4, 79)_C11', '(4, 79)_C12', '(4, 79)_C13', '(4, 79)_C14', '(4, 79)_C15']
 * problem: (4, 8)
 * Graphs:
 - ['(4, 8)_C8', '(4, 8)_C9', '(4, 8)_C10', '(4, 8)_C11', '(4, 8)_C12', '(4, 8)_C13', '(4, 8)_C14', '(4, 8)_C15']
dataset length: 2


## Using `Graph_Builder` 

In order to use the graph builder and build graphs while training on the fly, we need to use the `RavenDataset` and `raven_flat_collate` based dataloader.

PS: This method is time consuming, it is used for debugging and testing

In [8]:
flat_crops                = batch[0] # crops               ,shape: [xxx, 1, 80, 80]
flat_attributes           = batch[1] # attributes          ,shape: [xxx, 4]
flat_bboxes               = batch[2] # bbox                ,shape: [xxx, 4] 
flat_panel_indices        = batch[3] # panel indices       ,shape: [xxx]
flat_component_indices    = batch[4] # component indices   ,shape: [xxx]
flat_config_indices       = batch[5] # config indices      ,shape: [xxx]
flat_problem_indices      = batch[6] # problems indices    ,shape: [xxx]
flat_problem_targets      = batch[7] # problems targets    ,shape: [xxx]
flat_batch_indices        = batch[8] # batch indices       ,shape: [xxx]

### RavenGnnModel 

Contains two main parts: 
- RavenGnnEncoder
- CandidateScorer

#### 1 RavenGnnEncoder

Based on the `GINEConv` message passing module and the `HeteroConv` module.

 - The `GINEConv` uses A neural network `h_(Theta)` that maps node features `x` of shape `[-1, in_channels]` to shape `[-1, out_channels]`
<div align="center">
  <img src="../assets/images/GINEConv_math.png" alt="Alt text" style="width: 65%;">
</div>

In [9]:
from itertools import islice
from hri_gnn.models import RavenGnnEncoder
from hri_gnn.config.const import GRAPH_EDGE_TYPES, GRAPH_NODE_TYPES
from hri_gnn.models.gnn_encoder import CandidateScorer, score_and_loss_for_batch

active_edge_types = None
hidden_dim = 64 
num_layers = 3
dropout = 0.1

gnn_encoder = RavenGnnEncoder(GRAPH_NODE_TYPES, hidden_dim, num_layers, active_edge_types, dropout)

x_dict = {'entity': pyg_batch['entity'].x, 'layout': pyg_batch['layout'].x}
edge_index_dict = {edge_type: pyg_batch[edge_type].edge_index for edge_type in GRAPH_EDGE_TYPES.keys()}
edge_attr_dict = {edge_type: pyg_batch[edge_type].edge_attr for edge_type in GRAPH_EDGE_TYPES.keys()}

h_dict = gnn_encoder(x_dict, edge_index_dict, edge_attr_dict)
print(f"Example of the Origninal layout representation is: \n{x_dict["layout"][0]} \n shape {x_dict["layout"][0].shape}")
print(f"Example of the new layout node representation afeter {num_layers} is :\n{h_dict["layout"][0]} \n shape {h_dict["layout"][0].shape}")


print(f"- targets for every graph in the batch:\n{pyg_batch.y}")

Example of the Origninal layout representation is: 
tensor([  7.7260,  91.6392,   0.0000,   0.0000,  81.7939,   8.1895,   0.0000,
         67.5683, 105.1489,   0.0000,   0.3240,  33.4878,   0.0000, 102.0202,
          0.0000,  28.2877,   0.0000,  22.4428,   0.0000,  35.9425,   0.0000,
          0.0000,  60.9301,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
         39.0205,  36.9416,  33.5090,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,  50.8125,   0.0000,  40.9894,   0.0000,   0.0000,
         29.6623,  40.2300,   0.0000,   0.0000,   0.0000, 107.7555,   0.0000,
         28.8779,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,  31.5905,   0.0000, 118.9846,   0.0000,   0.0000,
         14.2286, -34.6651,  13.1872,  19.8484,  14.2261,   8.4755,  -8.7374,
        -23.0555, -18.3545,  -2.1088,  12.8012,  19.4503,  27.5327,  36.3298,
         29.2221,  18.5096,  -1.2195,  -4.9777, -17.5162, -33.6851, -46.1157,
        -53.

#### 2. CandidateScorer

baseed on a `BatchedAttentionPooling` to calculate scores, each problem have 8 candidates each candidate may have 1 or 2 layouts (based on the config): 
 - For each candidate we calculate a score (`Sequential(Linear, ReLU, Linear) 64 --> 1`)
  * For configs with 2 layouts per candidate an attention mechanisme based on learned weights is considered: each representation -> `Linear(64,1)` --> each couple `softmax` --> weighs*representations  --> 8 scores per problem 

In [10]:
scorer  = CandidateScorer(hidden_dim)

all_scores = scorer(h_dict['layout'], all_layout_keys, pyg_batch['layout'].batch)

print(f"All scores: \n {all_scores}")

All scores: 
 tensor([-0.3201, -0.2564, -0.3165, -0.3193, -0.2388, -0.3609, -0.3812, -0.2713,
        -0.2852, -0.1849, -0.2767, -0.1976, -0.3082, -0.2938, -0.2473, -0.1583],
       grad_fn=<SqueezeBackward1>)


In [11]:
scorer  = CandidateScorer(hidden_dim)

scores = score_and_loss_for_batch(h_dict['layout'], pyg_batch['layout'].batch, all_layout_keys,
                                            pyg_batch.y, scorer, range(8, 16) , n_problems)
print(f"- Scores per problem: \n{scores[0]}")
print(f"- Losses per problem: \n{scores[1]}")
print(f"- Mean loss: \n{scores[2]}")


- Scores per problem: 
[tensor([-0.1354, -0.1313, -0.0554, -0.1004, -0.1569, -0.1474, -0.1215, -0.0968],
       grad_fn=<SelectBackward0>), tensor([-0.1054, -0.0981,  0.0318, -0.0349, -0.0459, -0.1147, -0.0176, -0.1816],
       grad_fn=<SelectBackward0>)]
- Losses per problem: 
[tensor(2.0971, grad_fn=<NllLossBackward0>), tensor(2.1253, grad_fn=<NllLossBackward0>)]
- Mean loss: 
2.1112380027770996


In [12]:
from hri_gnn.models import build_hetero_data_batch, build_pyg_batch

hetero_data_dict, meta_dict = build_hetero_data_batch(batch, raven_encoder, include_columns= True, candidate_ids=range(8,16), device= None)
pyg_batch, batched_layout_keys, keys_order = build_pyg_batch(hetero_data_dict, meta_dict)

In [13]:
print(f"number of graphs in the batch: {len(hetero_data_dict)}\n")
print(f"graph names: {hetero_data_dict.keys()}\n")
problem = '(4, 8, 1)'
candidate = 'c10'
graph = '(4, 8, 1)'+'_'+ 'c10'
hetero_data = hetero_data_dict[graph]

print(f"hetero_data entity x shape:\n{hetero_data['entity'].x.shape}\n")
print(f"hetero_data layout x shape:\n{hetero_data['layout'].x.shape}\n")

print(f"pyg_batch layout x shape:\n{pyg_batch['layout'].x.shape}\n")

layout_contains_entity = hetero_data[('layout', 'contains', 'entity')]
entity_entity_intra_panel = hetero_data[('entity', 'intra_panel', 'entity')]
entity_entity_inter_panel = hetero_data[('entity', 'inter_panel_entity', 'entity')]
layout_layout_inter_panel = hetero_data[('layout', 'inter_panel_layout', 'layout')]
entity_belongs_to_layout = hetero_data[('entity', 'belongs_to', 'layout')]
print(f"hetero_data edges:\n")
print(f" - ('layout', 'contains', 'entity'): \n\t edge index shape:{layout_contains_entity.edge_index.shape}\n\t edge attr shape: {layout_contains_entity.edge_attr.shape}")
print(f" - ('entity', 'intra_panel', 'entity'): \n\t edge index: {entity_entity_intra_panel.edge_index.shape}\n\t edge attr shape: {entity_entity_intra_panel.edge_attr.shape}")
print(f" - ('entity', 'inter_panel_entity', 'entity'): \n\t edge index: {entity_entity_inter_panel.edge_index.shape}\n\t edge attr shape: {entity_entity_inter_panel.edge_attr.shape}")
print(f" - ('layout', 'inter_panel_layout', 'layout'): \n\t edge index: {layout_layout_inter_panel.edge_index.shape}\n\t edge attr shape: {layout_layout_inter_panel.edge_attr.shape}")
print(f" - ('entity', 'belongs_to', 'layout'): \n\t edge index :{entity_belongs_to_layout.edge_index.shape}\n\t edge attr shape: {entity_belongs_to_layout.edge_attr.shape}")

number of graphs in the batch: 16

graph names: dict_keys(['(4, 79, 0)_c8', '(4, 79, 0)_c9', '(4, 79, 0)_c10', '(4, 79, 0)_c11', '(4, 79, 0)_c12', '(4, 79, 0)_c13', '(4, 79, 0)_c14', '(4, 79, 0)_c15', '(4, 8, 1)_c8', '(4, 8, 1)_c9', '(4, 8, 1)_c10', '(4, 8, 1)_c11', '(4, 8, 1)_c12', '(4, 8, 1)_c13', '(4, 8, 1)_c14', '(4, 8, 1)_c15'])

hetero_data entity x shape:
torch.Size([27, 86])

hetero_data layout x shape:
torch.Size([18, 97])

pyg_batch layout x shape:
torch.Size([288, 97])

hetero_data edges:

 - ('layout', 'contains', 'entity'): 
	 edge index shape:torch.Size([2, 27])
	 edge attr shape: torch.Size([27, 1])
 - ('entity', 'intra_panel', 'entity'): 
	 edge index: torch.Size([2, 26])
	 edge attr shape: torch.Size([26, 2])
 - ('entity', 'inter_panel_entity', 'entity'): 
	 edge index: torch.Size([2, 80])
	 edge attr shape: torch.Size([80, 23])
 - ('layout', 'inter_panel_layout', 'layout'): 
	 edge index: torch.Size([2, 72])
	 edge attr shape: torch.Size([72, 97])
 - ('entity', 'belon

### ***RavenGnnModel***

In [14]:
from hri_gnn.metrics import gnn_predictions
from hri_gnn.models import RavenGnnModel
from hri_gnn.metrics import gnn_predictions
from hri_gnn.config.const import GRAPH_NODE_TYPES

active_edge_types = None
hidden_dim = 64 
num_layers = 3
dropout = 0.1

raven_gnn_model = RavenGnnModel(in_dims=GRAPH_NODE_TYPES, 
                                hidden_dim=hidden_dim, 
                                num_layers=num_layers, 
                                active_edge_types=active_edge_types, 
                                candidate_ids=range(8,16), 
                                dropout=dropout)

scores_per_problem, losses_per_problem, mean_loss = raven_gnn_model(pyg_batch, batched_layout_keys, n_problems)

print(scores_per_problem, losses_per_problem, mean_loss)

print(gnn_predictions(scores_per_problem, pyg_batch.y ,range(8,16)))


[tensor([ 0.1879, -0.0322, -0.0984, -0.1000, -0.0428, -0.0446, -0.1478, -0.1731],
       grad_fn=<SelectBackward0>), tensor([ 0.0420, -0.0376, -0.0270, -0.1425,  0.0313,  0.0254, -0.1114, -0.0316],
       grad_fn=<SelectBackward0>)] [tensor(1.8408, grad_fn=<NllLossBackward0>), tensor(2.0246, grad_fn=<NllLossBackward0>)] tensor(1.9327, grad_fn=<MeanBackward0>)
(tensor([0, 0]), tensor([0, 5]))


In [6]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
from hri_gnn.trainers.train_gnn import run_training, print_validation_results

# DATASET
DATASET_NAME = 'raven-20000'
PERCENTAGE_LIMIT = 0.75

# DATALOADER
BATCH_SIZE = 512

# TRAINER
NB_EPOCHS = 150
CHECK_FREQUENCY = 10

# GNNMODEL
HIDDEN_DIM = 64
NUM_LAYERS = 3
DROPOUT = 0.1

# OPTIMIZER
LEARNING_RATE= 5e-4
WEIGHT_DECAY = 1e-4

# SCHEDULER
MODE= 'min'
FACTOR=0.5
PATIENCE= 10
MIN_LEARNING_RATE = 1e-5

history, validation_results ,model_training_time = run_training(DATASET_NAME,
                                                                PERCENTAGE_LIMIT,
                                                                BATCH_SIZE,
                                                                NB_EPOCHS,
                                                                CHECK_FREQUENCY,
                                                                HIDDEN_DIM,
                                                                NUM_LAYERS,
                                                                DROPOUT,
                                                                LEARNING_RATE,
                                                                WEIGHT_DECAY,
                                                                MODE,
                                                                FACTOR,
                                                                PATIENCE,
                                                                MIN_LEARNING_RATE,
                                                                session_name='gine',
                                                                resume=True)
                                                                

Device : cuda
 GPU : NVIDIA GeForce RTX 5060 Laptop GPU

Loading dataset : raven-20000
  Percentage to use : 75.0%

Batch size : 512
Train dataset loaded : 61062 problems
Val dataset loaded   : 20356 problems
Test dataset loaded  : 20358 problems

[RESUME] Found checkpoint at epoch 100: C:\Users\dhias\workspaces\these\HRI-GNN\outputs\checkpoints\gnn\gnn_model_gine_checkpoint_epoch_100.pth
[RESUME] Resuming from epoch 100. Remaining: 50 epochs.
start training model: RavenGnnModel...


  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/50 - Train Loss: 0.8838 | Train Acc: 0.6265 | Test Loss: 0.7423 | Test Acc: 0.6902 | Test P: 0.6902 | Test R: 0.6902 | Test F1: 0.6902 | LR: 5.00e-04 ★
Epoch 2/50 - Train Loss: 0.8768 | Train Acc: 0.6312 | Test Loss: 0.7498 | Test Acc: 0.6846 | Test P: 0.6847 | Test R: 0.6846 | Test F1: 0.6846 | LR: 5.00e-04
Epoch 3/50 - Train Loss: 0.8677 | Train Acc: 0.6316 | Test Loss: 0.7426 | Test Acc: 0.6920 | Test P: 0.6920 | Test R: 0.6920 | Test F1: 0.6920 | LR: 5.00e-04 ★
Epoch 4/50 - Train Loss: 0.8641 | Train Acc: 0.6339 | Test Loss: 0.7396 | Test Acc: 0.6890 | Test P: 0.6890 | Test R: 0.6890 | Test F1: 0.6890 | LR: 5.00e-04
Epoch 5/50 - Train Loss: 0.8574 | Train Acc: 0.6391 | Test Loss: 0.7257 | Test Acc: 0.6960 | Test P: 0.6960 | Test R: 0.6960 | Test F1: 0.6959 | LR: 5.00e-04 ★
Epoch 6/50 - Train Loss: 0.8608 | Train Acc: 0.6363 | Test Loss: 0.7240 | Test Acc: 0.6944 | Test P: 0.6944 | Test R: 0.6944 | Test F1: 0.6944 | LR: 5.00e-04
Epoch 7/50 - Train Loss: 0.8617 | Train Acc: 0

Validating...:   0%|          | 0/40 [00:11<?, ?it/s]


Validation finished!....
Number of problems: 20356 | Number of batches: 40
Accuracy: 0.7376
Precision: 0.7376 | Recall: 0.7376 | F1: 0.7375


  RESULTS ON VALIDATION SET  
 Number: 
  - Problems : 20356 
  - Batches  : 40
  Accuracy  : 73.76%
  Precision : 0.7376
  Recall    : 0.7376
  F1 score  : 0.7375

[OK] Validation results saved : C:\Users\dhias\workspaces\these\HRI-GNN\outputs\history\gnn\validation_results.json

[OK] Training history saved : C:\Users\dhias\workspaces\these\HRI-GNN\outputs\history\gnn\training_history_gine.json
